In [1]:
import logging
import os
import sys
from datetime import datetime
from pprint import pprint
import json
import pandas as pd
from typing import List, Dict, Any
from langchain.schema import Document

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

from src import YouTubeAnalysisManager
from src import YouTubeDBSetup

In [2]:
yt = YouTubeDBSetup()
result = yt.get_all_comments()

📊 10,000개 댓글 로드 완료...
📊 20,000개 댓글 로드 완료...
✅ 총 21,989개 댓글 조회 완료!


In [4]:
cats = []
for r in result:
    # print(r['categories'])
    cats.extend(r['categories'])

In [5]:
import collections

counter = collections.Counter(cats)
# pprint(counter)

In [6]:
dict(counter)

total_sum = 0
hate_sum = 0
for k, v in counter.items():
    total_sum += v
    if k != '정체성' and k != '욕설' and k != '기타' and k != '연령' and k != '성별':
        hate_sum += v
hate_sum = total_sum - hate_sum
print(f"전체 {total_sum:>6}개")
print(f"혐오 {hate_sum:>6}개 {hate_sum/total_sum*100:>6.2f}%")
print()

# 각 카테고리별 통계 출력
categories = ['정체성', '욕설', '기타', '연령', '성별']


print(f"{'구분':>4} {'건수':>6} {'전체비율':>6} {'혐오비율':>6}")
print("=" * 30)
for category in categories:
    count = counter[category]
    total_percentage = count / total_sum * 100
    hate_percentage = count / hate_sum * 100 if hate_sum > 0 else 0
    
    print(f"{category:>4} {count:>6}개 {total_percentage:>6.2f}% {hate_percentage:>6.2f}%")


전체  22617개
혐오   3360개  14.86%

  구분     건수   전체비율   혐오비율
 정체성   2414개  10.67%  71.85%
  욕설    542개   2.40%  16.13%
  기타    309개   1.37%   9.20%
  연령     51개   0.23%   1.52%
  성별     44개   0.19%   1.31%


In [8]:
analysis = YouTubeAnalysisManager()
result = analysis.comment_classifier.classify_single_comment("아구 진짜로 찢재명 간첩들 이걸 어찌 잡아야 하나요<br>선관위 폭파시켜라")

기존 벡터스토어 로드 완료
리트리버 초기화 완료: basic (k=5)
LLM Service Provider: openai
OpenAI LLM 서비스가 'gpt-5-mini' 모델로 초기화되었습니다.
gpt-5-mini
1. (Isolated) Query Embedding: 0.8960 초
2. Full Retrieval (Embedding + Search): 0.7444 초
3. Formatting: 0.0000 초
4. Prompt Generation: 0.0021 초
content='{\n  "prompt": "주어진 정의와 분류 기준에 따라 텍스트가 혐오표현인지 여부와 해당 카테고리(성별, 연령, 정체성, 욕설, 기타, 혐오없음)를 결정하십시오. 정치적 비판/조롱은 보호 속성과 무관하면 \'기타\'로 분류합니다.",\n  "input_text": "\\"아구 진짜로 찢재명 간첩들 이걸 어찌 잡아야 하나요<br>선관위 폭파시켜라\\"",\n  "is_hate_speech": true,\n  "categories": [\n    "기타"\n  ],\n  "reasoning": "문장은 특정 정치인(또는 그 지지자/관련 집단)을 \'간첩\'이라고 모욕·비방하면서(모욕/비하), 이어서 \'선관위 폭파시켜라\'라고 명백한 폭력·파괴 행위 선동을 포함합니다. 대상이 성별·연령·인종 등 보호 속성에 근거하지 않는 정치적 공격이므로 카테고리는 \'기타\'로 분류합니다. 욕설(직접적인 비속어)은 포함되어 있지 않으나 폭력 선동과 모욕성이 뚜렷하여 혐오·유해 표현으로 판단됩니다.",\n  "similar_cases_used": [],\n  "target_group": "이재명 관련 인물/이재명 지지자 등 특정 정치 집단",\n  "hate_type": "모욕/비하 및 폭력 선동 (선동적 폭력·테러 행위 권유)"\n}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tok

In [14]:
pprint(result.reasoning)



("문장은 특정 정치인(또는 그 지지자/관련 집단)을 '간첩'이라고 모욕·비방하면서(모욕/비하), 이어서 '선관위 폭파시켜라'라고 명백한 "
 "폭력·파괴 행위 선동을 포함합니다. 대상이 성별·연령·인종 등 보호 속성에 근거하지 않는 정치적 공격이므로 카테고리는 '기타'로 "
 '분류합니다. 욕설(직접적인 비속어)은 포함되어 있지 않으나 폭력 선동과 모욕성이 뚜렷하여 혐오·유해 표현으로 판단됩니다.')
